## Import Libraries

In [ ]:
# Import necessary libraries for data handling, optimization, and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import time
import pickle
from itertools import permutations
from gurobipy import *
from gurobipy import GRB

In [ ]:
# Set your working directory
my_folder = "C:/Users/hyunwoolee/OneDrive - Virginia Tech/Hyunwoo Research/upload"

In [ ]:
SEED = 11
random.seed(SEED)
np.random.seed(SEED)

## General Functions

In [ ]:
# Converts a set of selected lakes into selfish and global utility values.
# - 'selected_lakes' is a list of lake indices selected by the players (counties).
# - Returns: 
#     1. A dictionary of selfish utilities per county.
#     2. A scalar value representing the global utility (social welfare).

def from_x_to_obj(selected_lakes):
    selfish_obj = {county: 0.0 for county in counties}
    global_obj = 0.0

    for (u, v) in arcs:
        if (u in selected_lakes) or (v in selected_lakes):     # y_arc = 1
            val = n[(u, v)] * t[(u, v)]
            global_obj += val
            # add to the destination county of v
            county_v = lake_county[v]
            selfish_obj[county_v] += val

    return selfish_obj, global_obj

In [ ]:
def ebmc_player_utility(selected_set, county):
    # selected_set is a set of selected lakes
    return sum(
        n[arc] * t[arc]
        for arc in (arcs_c[county] + arcs_plus_c[county])
        if (arc[0] in selected_set) or (arc[1] in selected_set)
    )


In [ ]:
def printSolution(model, x, selected_lakes):
    
    # Check if the model found a feasible solution
    if model.SolCount > 0:
        print('\nObjective value: %g' % model.ObjVal)

        # Append selected lakes (x[i] ≈ 1) to the result list
        for i in I:
            if x[i].X > 0.1:
                # print('Lake %s: %g' % (i, x[i].X))
                selected_lakes.append(i)
    else:
        pass  # No feasible solution found


## Best-response problem

In [ ]:
def selfish(county):
    model = Model("Selfish")
    model.Params.LogToConsole = 0
    model.Params.OutputFlag = 0
    model.Params.MIPGap = 1e-6
    model.Params.Threads = user_Threads
    model.ModelSense = GRB.MAXIMIZE  # important

    # variables
    x = model.addVars(I_c[county] + I_c_complement[county], vtype=GRB.BINARY, name="x")

    relevant_arcs = [arc for arc in arcs if arc[1] in I_c[county]]
    y = model.addVars(relevant_arcs, vtype=GRB.BINARY, name="y")

    # constraints
    model.addConstr(quicksum(x[i] for i in I_c[county]) <= county_budget[county], name="Budget")
    model.addConstrs((y[arc] <= x[arc[0]] + x[arc[1]] for arc in relevant_arcs), name="Link")

    # Objective: maximize selfish utility for this county
    model.setObjective(quicksum(t[arc] * n[arc] * y[arc] for arc in relevant_arcs), GRB.MAXIMIZE)
    
    # store handles for fast updates later
    model._x = x
    model._relevant_arcs = relevant_arcs
    model.update()
    return model

## RRR-BRD algorithm

In [ ]:
def RRR_BRD(x_current, selfish_model, max_init, max_iteration):
    """
    Executes the Round-based Random Restart Best-Response Dynamics (RRR-BRD) algorithm for the EBMC game.

    Starting from an initial strategy profile (given as a flat list of selected lakes),
    this function iteratively updates each county's strategy using best-response optimization.
    For each restart:
      - generate a randomized feasible profile,
      - run BRD with randomized county order every round.

    Args:
        x_current (list): Initial strategy profile as a flat list of selected lakes.
        selfish_model (dict or None): Pre-built Gurobi models for each county's best-response problem.
                                      If None, models are created on demand and reused over restarts.
        max_init (int): Max number of random initial profiles (restarts).
        max_iteration (int): Max number of BRD iterations per restart.

    Returns:
        selected_lakes (dict[int -> dict[county -> list]]): BRD path; selection per iteration and county.
        BR_PNE_found (bool): True if a Pure Nash Equilibrium was found.
        total_iterations (int): Total best-response updates performed.
        solution_last (dict[county -> float]): County utilities at the last iteration.
    """

    global BR_PNE_found, I_c

    TOL = 1e-8

    # Initialize selected_lakes and solution dictionary
    selected_lakes = {0: {}}
    for county in counties:
        selected_lakes[0][county] = [i for i in x_current if lake_county[i] == county]

    solution = {0: from_x_to_obj(x_current)[0]}  # dict[county -> utility] at round 0

    # Create a container for selfish models if not already provided
    if selfish_model is None:
        selfish_model = {}
    
    # Initial settings
    playing_sequence = counties.copy()
    random.shuffle(playing_sequence)
    total_iterations = 0

    # Try up to max_init different initial strategy profiles
    for num_init in range(1,max_init+1):
        if num_init > 1:
            # Generate a new random strategy profile
            sel_dict = generate_feasible_solution_EBMC_alternating(num_init, I_c, county_budget)
            x_current = ebmc_flatten_selection(sel_dict)

            # Reset initial selections and utilities            
            selected_lakes = {0: {}}
            for county in counties:
                selected_lakes[0][county] = [i for i in x_current if lake_county[i] == county]

            solution = {0: from_x_to_obj(x_current)[0]}

        # Run best-response updates up to max_iteration times
        for num in range(1,max_iteration+1):
            total_iterations += 1

            # start this round from previous profile
            selected_lakes[num] = {county: selected_lakes[num - 1][county] for county in counties}
            solution[num] = {county: 0.0 for county in counties}


            for county in playing_sequence:
                if county not in selfish_model:
                    selfish_model[county] = selfish(county)
                m = selfish_model[county]

                # current full profile
                current_profile_flat = [lake for c, lakes in selected_lakes[num].items() for lake in lakes]

                # incumbent utility
                u_inc = float(ebmc_player_utility(current_profile_flat, county))

                # fix complements
                for i in I_c_complement[county]:
                    var = m._x[i]
                    if i in current_profile_flat:
                        var.lb, var.ub = 1, 1
                    else:
                        var.lb, var.ub = 0, 0

                incumbent_self = selected_lakes[num][county]
                for i in I_c[county]:
                    m._x[i].start = 1 if i in incumbent_self else 0

                m.update()
                m.optimize()

                u_br = float(m.ObjVal)

                if u_br > u_inc + TOL:
                    selected_lakes[num][county] = [i for i in I_c[county] if m._x[i].X > 0.5]
                    solution[num][county] = u_br
                else:
                    selected_lakes[num][county] = incumbent_self
                    solution[num][county] = u_inc

            # Check for convergence to a Pure Nash Equilibrium
            if selected_lakes[num] == selected_lakes[num-1]:
                # print(' PNE found at %d iteration ================================================================== '%(num))
                BR_PNE_found = True
                return selected_lakes, BR_PNE_found, total_iterations, solution[num]  
        
            # randomize next round order
            playing_sequence = counties.copy()
            random.shuffle(playing_sequence)

    # No equilibrium found after all initializations and iterations        
    return selected_lakes, False, total_iterations, {county:0 for county in counties}

## Experiment Loop: Run All Settings for Random Dataset

In [ ]:
def ebmc_flatten_selection(sel_by_county):
    return [lake for _, lakes in sel_by_county.items() for lake in lakes]


def ebmc_br_value(county, profile_dict, selfish_model):
    """Solve county BR against end-of-round profile_dict (others fixed)."""
    if county not in selfish_model:
        selfish_model[county] = selfish(county)
    m = selfish_model[county]

    # lakes selected by other counties
    selected_other = set()
    for c, lakes in profile_dict.items():
        if c != county:
            selected_other.update(lakes)

    # fix complement vars (all lakes not owned by this county)
    for i in I_c_complement[county]:
        var = m._x[i]
        if i in selected_other:
            var.lb, var.ub = 1, 1
        else:
            var.lb, var.ub = 0, 0

    # warm start own vars
    selected_self = set(profile_dict[county])
    for i in I_c[county]:
        m._x[i].start = 1 if i in selected_self else 0

    m.update()
    m.optimize()
    return float(m.ObjVal)


def ebmc_alpha_end_of_rounds(selected_lakes_hist, max_round=20, selfish_model=None):
    if selfish_model is None:
        selfish_model = {}

    R = min(max_round, max(selected_lakes_hist.keys()))
    alpha_by_round = {}
    alpha_i_by_round = {}

    best_alpha = float("inf")
    best_round = None
    best_profile = None
    best_global_obj = None

    for t in range(1, R + 1):
        profile = selected_lakes_hist[t]  # end-of-round profile

        # current utilities + global objective under this profile
        x_flat = ebmc_flatten_selection(profile)
        u_curr, global_obj = from_x_to_obj(x_flat)

        ratios = {}
        for county in counties:
            br = ebmc_br_value(county, profile, selfish_model)
            ratios[county] = br / float(u_curr[county])  # assumes > 0

        alpha_t = max(ratios.values())
        alpha_by_round[t] = alpha_t
        alpha_i_by_round[t] = ratios

        if alpha_t < best_alpha:
            best_alpha = alpha_t
            best_round = t
            best_profile = profile
            best_global_obj = float(global_obj)

    return alpha_by_round, alpha_i_by_round, best_alpha, best_round, best_profile, best_global_obj


In [ ]:
non_PNE_instances = [('multi', 15, 50, 0.8), ('multi', 20, 50, 0.5),  ('multi', 25, 50, 0.8), ('multi', 30, 50, 0.8)]

standard_time_limit = 1800
user_Threads = 16
global type_dataset 


for inst in non_PNE_instances:
    (type_dataset, county_size, num_lakes_per_county, budget_ratio) = inst

    # === Load Dataset === #
    df_edge = pd.read_csv(my_folder+f"/BZR_EBMC/EBMC_generated/{type_dataset}_dataset/{county_size}_{num_lakes_per_county}_{budget_ratio}.csv", index_col=0)
    with open(my_folder+f"/BZR_EBMC/EBMC_generated/{type_dataset}_dataset/info_data.pickle",'rb') as f:
        info_data = pickle.load(f)

    # === Unpack Experiment Settings === #
    counties = info_data[(county_size,num_lakes_per_county,budget_ratio)][0] 
    num_lakes_per_county = info_data[(county_size,num_lakes_per_county,budget_ratio)][1]
    infestation_status = info_data[(county_size,num_lakes_per_county,budget_ratio)][2]
    county_budget = info_data[(county_size,num_lakes_per_county,budget_ratio)][3]   
    infested_lakes = [key for key, values in infestation_status.items() if any(val > 0 for val in values.values())]

    lakes = np.unique(np.concatenate((df_edge['dow_origin'].unique(),df_edge['dow_destination'].unique()),0))
    lake_county = {lake: lake[:2] for lake in lakes}

    # === Compute Lake Weights === #

    w={}
    for i in lakes:
        w[i]=0
    for index,row in df_edge.iterrows():
        if row['bij'] != 0:
            w[row['dow_origin']] += row['bij']*row['weight']
            w[row['dow_destination']] += row['bij']*row['weight']

    # === Set Model Parameters === #

    I = lakes
    I_c = {county: [i for i in I if i[:2] == county] for county in counties}
    I_c_complement = {county: [i for i in I if i not in I_c[county]] for county in counties}

    arcs, n, t = [], {}, {}

    for index in df_edge.index:
        n[df_edge.loc[index,'dow_origin'], df_edge.loc[index,'dow_destination']] = df_edge.loc[index,'weight']
        t[df_edge.loc[index,'dow_origin'], df_edge.loc[index,'dow_destination']] = df_edge.loc[index,'bij']

    arcs = list(n.keys())

    # arcs_c means arcs within county c
    # arcs_plus_c means incoming arcs to county c
    # arcs_minus_c means outgoing arcs from county c

    arcs_c, arcs_plus_c, arcs_minus_c = {}, {}, {}

    for county in counties:
        arcs_c[county] = [arc for arc in arcs if (arc[0] in I_c[county]) and (arc[1] in I_c[county])]
        arcs_plus_c[county] = [arc for arc in arcs if (arc[1] in I_c[county]) and (arc[0] in I_c_complement[county])]
        arcs_minus_c[county] = [arc for arc in arcs if (arc[0] in I_c[county]) and (arc[1] in I_c_complement[county])]



    # === Run BRD (initial strategy profile: 0) === #
    print("="*35 + "  BRD(0)_model  " + "="*35)

    x_current = []
    start_time = time.time()

    hist, found, _, sol_last = RRR_BRD(x_current, selfish_model=None, max_init = 1, max_iteration=20)

    if found:
        # final profile is hist[max(hist.keys())]
        final_round = max(hist.keys())
        _, best_global = from_x_to_obj(hist[final_round])
        best_alpha = 1.0
        best_round = final_round
    else:
        alpha_round, alpha_i_round, best_alpha, best_round, best_prof, best_global = ebmc_alpha_end_of_rounds(hist, max_round=20)

    end_time = time.time()
    time_approx_BRD = end_time - start_time

    ##################################  Summarize Results ####################################
    # Create a DataFrame with specified indexes and columns
    columns = ['num_cts','num_lakes','num_infested_lakes','budget_ratio','budget',
               'approx_BRD_obj','best_alpha','approx_T','best_round']   

    # Create the DataFrame
    df_tmp_test = pd.DataFrame(index=['Total'], columns=columns)

    df_tmp_test.loc['Total','num_cts'] = len(counties)
    df_tmp_test.loc['Total','num_lakes'] = num_lakes_per_county*len(counties)
    df_tmp_test.loc['Total','num_infested_lakes'] = sum(len([lake for lake in infested_lakes if lake[:2] ==county]) for county in counties)
    df_tmp_test.loc['Total','budget_ratio'] = budget_ratio
    df_tmp_test.loc['Total','budget'] = sum(county_budget[county] for county in counties)
    df_tmp_test.loc['Total','approx_BRD_obj'] = best_global
    df_tmp_test.loc['Total','best_alpha'] = best_alpha
    df_tmp_test.loc['Total','approx_T'] = time_approx_BRD
    df_tmp_test.loc['Total','best_round'] = best_round

    # === Save Results to CSV === #
    results_path = f"{my_folder}/BZR_EBMC/EBMC_results/{type_dataset}_dataset/EBMC_approx_test.csv"
    try:
        df_test = pd.read_csv(results_path, index_col=0)
        df_test = pd.concat([df_test, df_tmp_test])
    except FileNotFoundError:
        df_test = df_tmp_test

    df_test.to_csv(results_path)